# 자연어 질문에서 Metadata Filter 생성하기
- 자연어 질문을 검색 조건으로 변환하고 변환 된 결과를 Pincone metadata filter로 직접 조립한다.

## 환경설정

In [1]:
from dotenv import load_dotenv

load_dotenv()

# PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_META_INDEX_NAME = 'adv-meta-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_DIMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [2]:
import pandas as pd

documents_df = pd.read_csv('data/documents_meta.csv')
queries_df = pd.read_csv('data/queries_meta.csv')

In [3]:
documents_df.head()

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",박민준,여행;제주;관광
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이서연,음식;비빔밥;역사
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",최유나,연예;음악;대중문화
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...,정하늘,역사;문화;문자
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...,정하늘,역사;군사;전쟁


In [4]:
queries_df.head()

,query_id,query_text,relevant_doc_ids
0,MQ1,김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서...,D27=3
1,MQ2,김철수 저자의 검색 카테고리 문서에서 Contextual Compression 개념...,D28=3
2,MQ3,김철수가 쓴 검색 카테고리 문서 중 Self-Query Retriever 원리를 다...,D29=3
3,MQ4,김철수 저자의 검색 카테고리 문서에서 Multi-Hop Retrieval 예시를 찾아줘,D30=3
4,MQ5,한지민 저자의 AI 카테고리 문서 중 기후 예측 연구 사례와 모델을 다룬 자료는?,D14=3;D26=2


## 벡터 스토어 준비

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings
)

## 출력 함수 

In [6]:
def print_docs(docs):
    for doc in docs:
        print(f'{doc.metadata["doc_id"]}: ')
        print('author:', doc.metadata.get('author'))
        print('category:', doc.metadata.get('category'))
        print(doc.page_content)
        print()

def doc_ids(docs):
    return [doc.metadata["doc_id"] for doc in docs]

## 자연어 질문을 구조화 된 검색 조건으로 변환하기

In [8]:
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_openai import ChatOpenAI

class MetadataSearchQuery(BaseModel):
    
    query: str = Field(
        description=(
            '벡터 검색에 사용할 핵심 검색어. '
            '작성자나 카테고리 조건은 제외하고, 문서 내용과 관련 된 검색어만 작성한다.'
        )
    )

    author: Optional[str] = Field(
        default=None,
        description=(
            '문서를 작성한 저자 이름. '
            '질문에 저자 조건이 없으면 null로 둔다. '
            '예: 김철수, 한지민, 이서연, 박민준, 최유나'
        )
    )

    categories: List[str] = Field(
        default_factory=list,
        description=(
            '문서가 속한 카테고리 조건 목록. '
            '질문에 카테고리 조건이 없으면 빈 리스트로 둔다. '
            '예: 검색, RAG, 벡터DB, AI, 음식, 여행'
        )
    )

llm = ChatOpenAI(model=OPENAI_LLM_MODEL, temperature=0)
structured_llm = llm.with_structured_output(MetadataSearchQuery)

## 검색 조건 생성 함수

In [9]:
def generate_search_query(user_query: str) -> MetadataSearchQuery:
    prompt = f'''
다음 사용자 질문을 벡터 검색 조건으로 변환하세요.

규칙:
1. query에는 문서 본문과 의미적으로 비교할 검색어만 작성하세요.
2. 작성자 조건은 author에 작성하세요.
3. 카테고리 조건은 categories에 작성하세요.
4. 질문에 작성자 조건이 없으면 author는 null로 두세요.
5. 질문에 카테고리 조건이 없으면 categories는 빈 리스트로 두세요.
6. '검색 카테고리', '카테고리가 검색'처럼 표현되면 categories에 '검색'을 넣으세요.
7. 'AI 관련 문서'처럼 표현 되면 categories에 'AI'를 넣으세요.

사용자 질문:
{user_query}
'''
    return structured_llm.invoke(prompt)

In [10]:
user_query = '김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘'

search_query = generate_search_query(user_query)
search_query

MetadataSearchQuery(query='ChromaDB Qdrant 비교', author='김철수', categories=['검색'])

## 구조화 결과를 Pinecone filter로 변환

In [11]:
def build_pinecone_filter(search_query: MetadataSearchQuery):
    conditions = []

    if search_query.author:
        conditions.append({'author' : {'$eq' : search_query.author}})

    if search_query.categories:
        conditions.append({'category' : {'$in' : search_query.categories}})

    if len(conditions) == 0:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {'$and' : conditions}    

In [12]:
filter_dict = build_pinecone_filter(search_query)
filter_dict

{'$and': [{'author': {'$eq': '김철수'}}, {'category': {'$in': ['검색']}}]}

## 자연어 기반 metadata filter 검색 함수

In [13]:
def natural_language_filter_search(user_query: str, top_k: int = 5, verbose: bool = True):
    search_query = generate_search_query(user_query)
    filter_dict = build_pinecone_filter(search_query)

    docs = vector_store.similiarity_search(
        search_query.query,
        k=top_k,
        filter=filter_dict
    )

    if verbose:
        print('원본 질문')
        print(user_query)
        print()
        print('생성 된 검색어')
        print(search_query.query)
        print()
        print('추출 된 author:')
        print(search_query.author)
        print()
        print('추출 된 categories:')
        print(search_query.categories)
        print()
        print('pinecone filter:')
        print(filter_dict)
        print()
        print('검색 결과 id:')
        print(doc_ids(docs))
        print('=' * 100)

    return search_query, filter_dict, docs